# Transformer 笔记

## 整体架构

<img src="./imgs/0.png" width=70%>

我们先建立一个根本的逻辑前提：

$QKV$ 矩阵可以具有物理意义(包括以后任何参数都可以具有物理意义),这个物理意义是由**损失函数**约束出来的

换句话讲，***只要你设计的损失函数能够起到对应的约束作用，你就可以赋予参数任何的物理意义***

## 输入数据

输入 = 词本身的语义向量（Word Embedding） + 词的位置向量（Positional Encoding）

<img src="./imgs/1.png" width=70%>

token并不只代表一个字，可以理解为能表意的最小单元。

比如：“一条”只有在两字合为一组词的时候才是最小表意单元，单独一个“一”或者“条”就无法作为最小的表意单元。

### 词嵌入(Word Embedding)



词嵌入层的作用就是把token的独热编码转换为语义向量，也称词向量

<img src="./imgs/2.png" width=70%>

token的独热编码与词嵌入矩阵相乘得到其对应的词向量

词嵌入矩阵的维度为$d \times V$,独热编码的维度为$V \times 1$,两者相乘后得到的词向量维度为$d \times 1$。$d$在此处的取值为512，远小于$V$

词嵌入的本质就是将token独热编码这一稀疏张量转化为词向量这一稠密张量，从而达到降维的目的

### 位置编码 (Positional Encoding) 

位置编码是对词嵌入层输出的词向量进行的操作

<img src="./imgs/3.png" width=70%>

其中：$pos$ 是词在句子中的位置索引（0, 1, 2...）。

$i$ 是词向量维度的索引（例如在 512 维的向量中，从 0 到 255）。

$d_{model}$ 是词向量的总维度（如 512）。

$2i$ 表示偶数维度用正弦函数， 

$2i+1$ 表示奇数维度用余弦函数。

最后的输入就为：

$$Input_1 = V_{词向量} + PE_1$$

## 自注意力机制

### 为什么不能使用$XX^{T}$

词向量在特征空间中会进行聚类，关系越紧密性质越相似的词向量夹角会越小

我们期望知晓词向量在特征空间中的相似关系，使用向量点积的形式来进行衡量，并通过点积的大小初步判断相似的程度。

为此我们将所有词向量并置组成矩阵$X$,与其转置$X^{T}$相乘就能初步得到词向量之间的点积

比如矩阵$XX^{T}$的第$i$行第$j$列就表示$X$中第$i$个词向量与第$j$个词向量的点积

<img src="./imgs/5.jpg" width=70%>

### 引入$QKV$矩阵

$QKV$矩阵的引入相当于给予模型可学习的参数，原始$XX^{T}$根本没有可学习的参数

<img src="./imgs/4.png" width=70%>

$d_{k}$ 的含义：

$$d_k = \frac{d_{model}}{h}$$

其中：

$d_{model}$：模型的总输入维度，也就是token的维度

$h$：多头注意力的头数

关于$Q$、$K$、$V$ 的理解:

你可以把注意力机制想象成去图书馆找书的过程：

$Q$（Query / 查询）：你大脑里的搜索意图。比如你想找“关于计算机视觉的书籍”。**在模型中，这是当前正在处理的词汇或特征的代表。**

$K$（Key / 键）：图书馆里每本书的标签或书名。比如“本书关于 PyTorch 与深度学习”。**在模型中，它是输入序列中每个词汇的特征标识。**

$V$（Value / 值）：书本里具体的正文内容。**在模型中，它是输入序列中每个词汇实际携带的语义信息。**

对于$QK^{T}$，模型将每个词转化为 $d_k$ 维空间(我们也叫特征空间)中的一个向量。$Q$ 和 $K^{T}$ 相乘，就是让当前词的 Query 向量去和句子中所有词的 Key 向量来计算点积。

点积越大，证明 Query 向量与 Key 向量夹角越小，两个词在特征空间中靠得越近，语义匹配度就越高，最后的注意力得分就高。

所以$QK^{T}$的物理意义就是：在当前需求(Query)下，每个token的特征与Query的匹配程度，**更近一步就是当前词汇与序列中每个词汇的匹配程度与关联程度。**

## 多头注意力机制

<img src="./imgs/7.png" width=70%>

相当于多个独立的自注意力机制，将结果concat

## 掩码注意力机制

<img src="./imgs/8.png" width=70%>

Transformer与全卷积神经网络一样都可以接收非固定尺寸的输入

有时一批次的数据(语句)长度不同，实际上Transformer是可以处理的，但不同长度最好按不同批次处理，这样方便GPU并行计算，训练效率更高

因此为了方便GPU并行运算，我们使用Padding来强制补齐使得每一批次的语句长度一致

## 层归一化(Layer Normalization)

### 层归一化(LayerNorm)与批归一化(BatchNorm)的区别

<img src="./imgs/9.png" width=70%>

## 文件操作

In [ ]:
import torch
import torch.nn as nn

# 此处使用的分词器的逻辑较为简单，基本依据语句中的空格分词

from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.trainers import WordLevelTrainer
from tokenizers.pre_tokenizers import Whitespace

from pathlib import Path

def get_or_build_tokenizer(config,ds,lang):

    tokenizer_path=Path(config['tokenizer_file'].format(lang))
    """
    config在此处是一个字典,用来存储模型配置,例如：

    config = {
        "tokenizer_file": "tokenizer_{0}.json",
        # ...其他参数
    }

    config['tokenizer_file']对应的值就是一个模板字符串,也就是:"tokenizer_{0}.json"

    {0}是占位符，用来后面填充变量

    .format()是字符串内置函数,其作用把变量 lang 填充进字符串模板的{0}位置

    假设lang='en',那么config['tokenizer_file'].format(lang)对应的值就是"tokenizer_en.json"

    Path("tokenizer_en.json")的作用就是将"tokenizer_en.json"转化成路径对象

    """

    if not Path.exists(tokenizer_path):
        tokenizer=Tokenizer(WordLevel(unk_token='[UNK]'))
        tokenizer.pre_tokenizer=Whitespace()
        trainer=WordLevelTrainer(show_progress=['[UNK]','[PAD]','[SOS]','[EOS]'],min_frequency=2)